In [25]:
import pandas as pd
import numpy as np
import json
import os
import zipfile
import glob
import time

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Using the already labelled files from Drive

In [28]:
root_path = '/content/drive/MyDrive/Research stuff/Final Year Design Project/Survey and collected data/mindseer_data/'

participants = 0
for name in glob.glob(root_path + '*.zip'): 
    participants += 1

print(participants)

104


In [29]:
#code snippet from https://stackoverflow.com/questions/50008296/facebook-json-badly-encoded

def parse_obj(obj):
    if isinstance(obj, str):
        return obj.encode('latin_1').decode('utf-8')

    if isinstance(obj, list):
        return [parse_obj(o) for o in obj]

    if isinstance(obj, dict):
        return {key: parse_obj(item) for key, item in obj.items()}

    return obj

In [30]:
def data_insert(path, id, data_list):
    try:
        f = open(path, 'r')
        try:
            raw_data = parse_obj(json.load(f))
            if (path[-6] == '1'):
                json_data = {}
                if type(raw_data) is dict:
                    json_data["posts"] = []
                    json_data["posts"].append(raw_data)
                else:
                    json_data["posts"] = raw_data
                    raw_data = json_data 
        except UnicodeDecodeError as error:
            print(error)
    except FileNotFoundError:  
        raw_data = {}
    finally:
        raw_data['participant_id'] = id
        data_list.append(raw_data)

In [31]:
comments_data = []
groups_data = []
posts_data = []

for i in range(participants):
    comments_path = root_path + str(i) + '/comments/comments.json'
    data_insert(comments_path, i, comments_data)

    groups_path = root_path + str(i) + '/groups/your_posts_and_comments_in_groups.json'
    data_insert(groups_path, i, groups_data)

    posts_path = root_path + str(i) + '/posts/your_posts_1.json'
    data_insert(posts_path, i, posts_data)

In [32]:
print(len(comments_data))
print(len(groups_data))
print(len(posts_data))

104
104
104


In [33]:
comment_texts = []
for i in range(participants):
    text = ""
    try:
        for comment_details in comments_data[i]['comments']:
            #print(comment_details)
            try:
                for data in comment_details['data']:
                    text += (data['comment']['comment'] + " ")
            except KeyError:
                continue
    except KeyError:
        continue
    
    finally:
        #print(text)
        comment_texts.append(text)
    

In [34]:
len(comment_texts)

104

In [43]:
print(comment_texts[0])

Wuuuvvv eeewwww mooouuuurrrrr Dhonnobad dost. <3 Thanks bro.  Thanks sensei. <3 Happy to help, man. I guess we are happier now as a couple. Hence the weight gain XD nah manacche na :P Tis okaaayyy....Of course I remember thiss <3 The irony is the last point is the least important in the actual learning process... Witcher koro plezz Thanks bro https://www.youtube.com/watch?v=_O1hM-k3aUY hahahaha Wait you mean the "Law of Gravity" was not approved by the International Court of Law? :O  Stop these NASA liars. The real story was Japan launched explosive canon balls that bombed Pearl Harbor. Because gravity is a lie. :3 Thanks bro Thanks bro! <3 Nothing special. Watching Sakasama no Patema. It's interesting :3 Thanks dost! Bhalo achish? Thanks bhaiyya Merci beaucoup! :3 Thank you so muchh!  hahahaha....be a bird and excrete from above :3 hain. chironidra ekdom D: If you look closely, the guy couldn't take a nap or anything properly. Look how he was gripping the rope with his left hand under

In [35]:
groups_texts = []
for i in range(participants):
    text = ""
    try:
        for post_comments in groups_data[i]["group_posts"]["activity_log_data"]:
            try:
                for data in post_comments['data']:
                    if 'post' in data:
                        if (text.find(data['post']) == -1): 
                            text += (data['post'] + " ")
                    elif 'comment' in data:
                        if (text.find(data['comment']['comment']) == -1):
                            text += (data['comment']['comment'] + " ")
            except KeyError:
                continue
    except KeyError:
        continue
    finally:
        #print(text)
        groups_texts.append(text)

In [36]:
len(groups_texts)

104

In [37]:
posts_texts = []
for i in range(participants):
    text = ""
    try:
        for post in posts_data[i]["posts"]:
            #print(post)
            try:
                for data in post['data']:
                    #print(data)
                    if 'post' in data:
                        text += (data['post'] + " ")  
            except KeyError:
                continue                 
    except KeyError:
        continue

    finally:
        posts_texts.append(text)



In [38]:
len(posts_texts)

104

In [39]:
all_text = []
for i in range(participants):
    text = comment_texts[i] + " " + groups_texts[i] + " " + posts_texts[i]
    #print(len(text))
    all_text.append(text)

print(len(all_text))


104


In [40]:
all_text_data = pd.DataFrame(all_text)
all_text_data.head()

,0
0,Wuuuvvv eeewwww mooouuuurrrrr Dhonnobad dost. ...
1,Nahian Rakib Its family for luffy Luffy's fami...
2,Rehnuma Rahman Raisa Sumaia Dhony In Shaa All...
3,Happy Birthday☺Rehnuma Rahman Raisa OK Sir Sir...
4,


In [41]:
all_text_data.shape

(104, 1)

In [42]:
#Saving to xlsx format
all_text_data.to_excel('/content/text_data.xlsx', sheet_name = 'Sheet 1', index = False, encoding = 'utf-8')